# 18 · Did any variable come from a real lookup table?
`17` found one real input: 2010 Census ZCTA population ÷ land area. Here 12 more variables are checked:
- `CAT_ZONE`, `QUOTED_AMT`, `PRIOR_CARRIER_GRP`, `CREDIT_SCORE`, `MAKE_MODEL`, `SAFETY_FEATURE_SCORE`;
- `ANNUAL_MILEAGE`, `COMMUTE_MILES_DAILY`;
- `MARITAL_STATUS`, `LIVING_STATUS`, `HIGH_EDU_IND`, `GENDER`.

Method:
1. Internal key dependence: how much of each variable each candidate key explains, against a permutation null (`lookup_tests.py`).
2. External comparisons only where structure exists.

In [1]:
import sys, json, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
sys.path.insert(0, '.')
import lookup_tests as LT, formula_steps as FS, geo_tests as G
from formula_steps import t
P, D, V = LT.tables()

## 1. Key dependence
- `explained` = η² (numeric) or Goodman–Kruskal τ (categorical) for the key; `null` = the same with the key shuffled (20 permutations).
- Rows shown: z > 4 and excess > 0.001.
- Calibration: `MONTHLY_PREMIUM` against `QUOTED_AMT` scores 1.0.

In [2]:
R = pd.concat([
    LT.screen(P, ['CAT_ZONE', 'QUOTED_AMT', 'PRIOR_CARRIER_GRP', 'CREDIT_SCORE'],
              ['STATE_CD', 'ZIP_CD', 'ZIP3', 'YEAR', 'NUM_DRIVERS', 'VEH_N_ROWS', 'COV_PKG_TYPE', 'PRIOR_CARRIER_GRP', 'CAT_ZONE', 'QUOTE_CHANNEL', 'DISCOUNT_IND', 'HOME_POLICY_IND', 'CREDIT_BAND'],
              categorical={'CAT_ZONE', 'PRIOR_CARRIER_GRP'}).assign(level='policy'),
    LT.screen(D, ['COMMUTE_MILES_DAILY', 'MARITAL_STATUS', 'LIVING_STATUS', 'HIGH_EDU_IND', 'GENDER'],
              ['STATE_CD', 'ZIP3', 'YEAR', 'DRIVER_AGE', 'AGE_BAND', 'GENDER', 'DRIVER_POS', 'NUM_DRIVERS', 'MARITAL_STATUS', 'LIVING_STATUS', 'HIGH_EDU_IND', 'ACC_FREE_BAND', 'TELEMATICS_OPT_IN'],
              categorical={'MARITAL_STATUS', 'LIVING_STATUS', 'HIGH_EDU_IND', 'GENDER'}).assign(level='driver'),
    LT.screen(V, ['ANNUAL_MILEAGE', 'MAKE_MODEL', 'SAFETY_FEATURE_SCORE'],
              ['STATE_CD', 'ZIP3', 'YEAR', 'MAKE_MODEL', 'VEH_AGE_KEY', 'OWNERSHIP_TYPE', 'COLOR', 'CAR_NO', 'GARAGE_IND', 'ANTI_THEFT_IND', 'VEH_N_ROWS'],
              categorical={'MAKE_MODEL'}).assign(level='vehicle')])
best = R.sort_values('excess', ascending=False).groupby('variable').head(1).set_index('variable')[['level', 'key', 'explained', 'z']]
print('strongest key per variable:'); display(best.round(4))
R[(R.z > 4) & (R.excess > 0.001)].sort_values(['variable', 'excess'], ascending=[True, False])[['variable', 'key', 'levels', 'explained', 'null', 'z']].style.format(precision=4).hide(axis='index')

strongest key per variable:


,level,key,explained,z
variable,,,,
CREDIT_SCORE,policy,CREDIT_BAND,0.9565,6535.6956
LIVING_STATUS,driver,AGE_BAND,0.5290,11563.2193
QUOTED_AMT,policy,STATE_CD,0.4987,5099.7948
HIGH_EDU_IND,driver,AGE_BAND,0.3683,6475.3942
CAT_ZONE,policy,COV_PKG_TYPE,0.1033,3341.0681
GENDER,driver,DRIVER_POS,0.0808,3916.5895
PRIOR_CARRIER_GRP,policy,STATE_CD,0.0303,810.5327
SAFETY_FEATURE_SCORE,vehicle,MAKE_MODEL,0.0088,1.1941
MARITAL_STATUS,driver,ZIP3,0.0019,2.1135


variable,key,levels,explained,null,z
CAT_ZONE,COV_PKG_TYPE,4,0.1033,0.0001,3341.0681
CAT_ZONE,CREDIT_BAND,6,0.0180,0.0001,371.7147
CAT_ZONE,STATE_CD,8,0.0095,0.0002,203.3027
CAT_ZONE,ZIP3,130,0.0127,0.0035,45.2001
CAT_ZONE,ZIP_CD,1169,0.0400,0.0321,12.1382
CREDIT_SCORE,CREDIT_BAND,6,0.9565,0.0002,6535.6956
CREDIT_SCORE,STATE_CD,8,0.2580,0.0002,3276.0470
CREDIT_SCORE,ZIP3,130,0.2583,0.0035,667.5981
CREDIT_SCORE,ZIP_CD,1169,0.2811,0.0319,235.3329
CREDIT_SCORE,COV_PKG_TYPE,4,0.2088,0.0001,4477.2438


**Reading:**
- **Independent draws** (no key explains more than chance): annual mileage, make/model, safety feature score, commute miles, marital status. They don't even vary with driver age, which real data always does.
- **Internal rules:** education, living status and gender depend on driver age and on the driver's position in the policy (§2).
- **State-linked chain:**
  - credit (26% explained by state);
  - package (credit explains 21%);
  - CAT zone (package explains 10%);
  - price (state 50%, credit band 34%);
  - prior carrier (state 3%).

## 2. Driver demographics are rules, not Census tables

In [3]:
ab = pd.cut(D.DRIVER_AGE, [15, 18, 20, 21, 25, 26, 35, 55, 75, 101])
display(pd.concat({'P(high edu)': D.groupby(ab, observed=True).HIGH_EDU_IND.mean(), 'P(dependent)': D.groupby(ab, observed=True).LIVING_STATUS.apply(lambda s: (s == 'dependent').mean()),
                   'P(own | not dependent)': D[D.LIVING_STATUS != 'dependent'].groupby(ab, observed=True).LIVING_STATUS.apply(lambda s: (s == 'own').mean()),
                   'P(married)': D.groupby(ab, observed=True).MARITAL_STATUS.apply(lambda s: (s == 'married').mean()), 'mean commute': D.groupby(ab, observed=True).COMMUTE_MILES_DAILY.mean()}, axis=1).round(3))
display(pd.crosstab(D.DRIVER_POS, D.GENDER, normalize='index').round(3))

,P(high edu),P(dependent),P(own | not dependent),P(married),mean commute
DRIVER_AGE,,,,,
"(15, 18]",0.0,1.000,0.000,0.496,22.494
"(18, 20]",0.0,1.000,0.000,0.494,22.724
"(20, 21]",0.619,1.000,NaN,0.496,23.111
"(21, 25]",0.657,0.999,1.000,0.536,22.330
"(25, 26]",0.63,0.304,0.656,0.457,22.174
"(26, 35]",0.633,0.002,0.553,0.491,22.690
"(35, 55]",0.62,0.000,0.547,0.496,22.562
"(55, 75]",0.628,0.000,0.542,0.501,22.565
"(75, 101]",0.619,0.000,0.552,0.495,21.856


GENDER,F,M
DRIVER_POS,,
1,0.357,0.643
2,0.695,0.305
3,0.501,0.499
4,0.505,0.495
5,0.512,0.488
6,0.478,0.522


**Reading:**
- **Education** is 0 up to age 20, then a flat ~62% at every age.
- **Living status:** everyone up to 25 is "dependent"; after that, own/rent is 55/45 regardless of age.
- **Marital status** is ~50% married at every age, 16–20 included.
- **Commute** is flat across ages.
- **Gender** alternates by position in the policy, a "spouse" pattern.

Real Census tables (ACS by age) look nothing like this, so these are hand-written rules with fixed probabilities.

## 3. State profiles are synthetic segment mixtures

In [4]:
bins = np.arange(400, 851, 50)
display(pd.DataFrame({s: np.histogram(g.CREDIT_SCORE.dropna().astype(float), bins)[0] / g.CREDIT_SCORE.notna().sum() for s, g in P.groupby('STATE_CD')}, index=bins[:-1])[['AL', 'GA', 'WI', 'NY', 'CT', 'FL', 'MN', 'NJ']].round(3))
P['CZ'] = pd.to_numeric(P.CAT_ZONE, errors='coerce')
P.groupby('STATE_CD').agg(credit_mean=('CREDIT_SCORE', 'mean'), median_price=('QUOTED_AMT', 'median'), cat_mean=('CZ', 'mean'), quotes=('POLICY_ID', 'size')).round(1).loc[['AL', 'GA', 'WI', 'NY', 'CT', 'FL', 'MN', 'NJ']]

,AL,GA,WI,NY,CT,FL,MN,NJ
400,0.003,0.006,0.006,0.002,0.001,0.000,0.000,0.000
450,0.084,0.085,0.087,0.039,0.003,0.009,0.006,0.006
500,0.385,0.332,0.346,0.150,0.035,0.039,0.024,0.027
550,0.373,0.348,0.372,0.201,0.098,0.108,0.075,0.073
600,0.128,0.133,0.134,0.247,0.263,0.276,0.229,0.219
650,0.025,0.062,0.043,0.217,0.280,0.282,0.230,0.238
700,0.001,0.020,0.009,0.092,0.188,0.166,0.236,0.220
750,0.001,0.015,0.004,0.048,0.119,0.111,0.181,0.199
800,0.000,0.000,0.000,0.004,0.014,0.009,0.021,0.019


,credit_mean,median_price,cat_mean,quotes
STATE_CD,,,,
AL,556.2,18311.0,3.4,778
GA,567.5,13810.0,3.4,2154
WI,560.7,14456.0,3.4,2241
NY,621.6,4911.0,3.0,11611
CT,669.0,2759.0,2.7,1825
FL,662.9,3026.0,2.7,8965
MN,683.8,2355.0,2.6,2010
NJ,684.2,2361.0,2.6,7287


**Reading:**
- Credit is a mix of three segments: low (~555), middle (~640) and high (~745), with state-specific weights:
  - AL and WI almost all low; GA low plus a small high tail;
  - NY half low, half middle;
  - CT/FL mostly middle; MN/NJ middle plus a large high group.
- Price and CAT zone follow the same grouping. AL/GA/WI have median prices of $14–18k vs $2.4–4.9k elsewhere, plus the high CAT distribution.
- Real state rankings differ: Wisconsin and Minnesota have among the highest average credit scores in the US, while Alabama, Georgia and Wisconsin are among the cheaper auto-insurance states.
- So these are synthetic state profiles, not real lookups.

## 4. Make/model list and the luxury flag

In [5]:
vp = json.load(open('../data/external/vpic_models_by_make.json')); ep = pd.read_parquet('../data/external/epa_vehicles_make_model_year.parquet')
ep['MAKE'], ep['MODEL'] = ep.make.str.upper().str.strip(), ep.model.str.upper().str.strip()
mm = V[['MAKE', 'MODEL']].drop_duplicates()
rows = []
for mk, g in mm.groupby('MAKE'):
    ours = set(g.MODEL.str.upper().str.strip())
    rows.append({'make': mk, 'models': len(ours), 'in NHTSA vPIC': len(ours & set(vp[mk])), 'in EPA fueleconomy': len(ours & set(ep[ep.MAKE == mk].MODEL))})
T = pd.DataFrame(rows).set_index('make'); T.loc['total'] = T.sum(); display(T)
print('examples:', sorted(mm[mm.MAKE == 'HONDA'].MODEL)[2:8], sorted(mm[mm.MAKE == 'MERCEDES-BENZ'].MODEL)[:4])
LUX = {'BMW', 'MERCEDES-BENZ', 'AUDI', 'CADILLAC', 'ACURA'}
w = V.assign(lux=V.MAKE.isin(LUX)).groupby('POLICY_ID').lux.agg(['mean', 'size']); w = w[w['size'] > 1]
print(f'multi-vehicle policies mixing luxury and non-luxury makes: {((w["mean"] > 0) & (w["mean"] < 1)).mean():.4f}')
g = V.groupby('MAKE_MODEL').SAFETY_FEATURE_SCORE.std(); print(f'safety feature score sd: within model (median) {g.median():.2f} vs overall {V.SAFETY_FEATURE_SCORE.std():.2f}')

,models,in NHTSA vPIC,in EPA fueleconomy
make,,,
ACURA,15,13,10
AUDI,33,19,26
BMW,196,64,79
BUICK,14,13,8
CADILLAC,16,14,12
CHEVROLET,89,39,23
CHRYSLER,18,16,11
DODGE,40,16,12
FORD,81,36,11


examples: ['AQUATRAX F-12', 'AQUATRAX F-12X', 'AQUATRAX F-12X GPSCAPE', 'AQUATRAX F-15', 'AQUATRAX F-15X', 'AQUATRAX F-15X GPSCAPE'] ['A180', 'A200', 'A250', 'A45 AMG']


multi-vehicle policies mixing luxury and non-luxury makes: 0.0000
safety feature score sd: within model (median) 1.89 vs overall 1.89


**Reading:**
- **The model list comes from a global vehicle-name list, not US data.** Only 38% of the 1,049 names are in NHTSA's vPIC and 32% in EPA's fuel-economy list. The list includes Honda AquaTrax jet skis and motorcycles, Canada-only Acuras (EL, CSX) and European-market Mercedes/BMW models (A180, 118i).
- **Luxury is exact:** the generator draws a luxury / non-luxury household, then makes from that group. No multi-vehicle policy mixes the two, and our 5-make list is exactly the generator's.
- **Safety feature score** is random per vehicle: it varies as much within a model as overall, so it isn't a rating lookup.

## 5. Price: rating factors + noise, and which part drives conversion
- Out-of-fold LightGBM for log price from all non-price columns. Price-derived engineered columns are excluded; including them leaked price and gave a spurious R² of 0.998.
- Then the conversion formula is tested with expected price, the residual, or both.

In [6]:
import lightgbm as lgb, formula as FM
from sklearn.model_selection import KFold
A = FM.prep(pd.concat([FS.lab, FS.hold])).reset_index(drop=True)
drop = {'POLICY_ID', 'SPLIT', 'CONVERT_IND', 'QUOTE_DT', 'ZIP_CD', 'ZIP3', 'QUOTED_AMT', 'MONTHLY_PREMIUM', 'QUOTED_AMT_MISSING', 'PRICE_PER_CREDIT', 'QUOTE_PER_DRIVER', 'QUOTE_PER_VEH'}
feats = [c for c in A if c not in drop and not c.startswith(('V2_', 'R_', 'DEN_', 'PKG_ORDINAL_IMP'))]
X = A[feats].copy()
for c in X:
    if not pd.api.types.is_numeric_dtype(X[c]): X[c] = X[c].astype('category')
ok = A.QUOTED_AMT.notna().to_numpy(); ly = np.log(A.QUOTED_AMT.astype(float)); oof = np.full(len(A), np.nan); imp = 0
for tr, va in KFold(5, shuffle=True, random_state=0).split(X):
    tt = tr[ok[tr]]; m = lgb.LGBMRegressor(n_estimators=1500, learning_rate=0.03, num_leaves=63, min_child_samples=10, verbose=-1).fit(X.iloc[tt], ly[tt])
    oof[va] = m.predict(X.iloc[va]); imp = imp + pd.Series(m.booster_.feature_importance('gain'), index=feats)
print(f'out-of-fold R² of log price: {1 - np.nanvar((ly - oof)[ok]) / np.var(ly[ok]):.3f}; residual sd {np.nanstd((ly - oof)[ok]):.3f}')
print('main price drivers (gain share):', (imp / imp.sum()).sort_values(ascending=False).head(5).round(3).to_dict())
pe = pd.Series(np.exp(oof), index=A.POLICY_ID); lr = pd.Series(ly - oof, index=A.POLICY_ID)
L = FS.L; L['P_EXP'] = pe.reindex(L.POLICY_ID).to_numpy(); L['LOGP_RESID'] = lr.reindex(L.POLICY_ID).to_numpy(); L['P_RESID_RAW'] = L.QUOTED_AMT.astype(float) - L.P_EXP
VAR = {'price → expected price only': G.swap(FS.SPEC8, 'QUOTED_AMT', 'P_EXP'), 'price → residual only': G.swap(FS.SPEC8, 'QUOTED_AMT', 'P_RESID_RAW'),
       '+ log residual per era (competitiveness)': FS.SPEC8 + [t('LOGP_RESID', 'linear', 'per-era')], '+ expected price per era': FS.SPEC8 + [t('P_EXP', 'linear', 'per-era')]}
rows = {}
for s in [(0,), (1, 2)]:
    b = FS.run4(FS.SPEC8, seeds=s)
    for k, v in VAR.items():
        r = FS.cmp(FS.run4(v, seeds=s), b); rows.setdefault(k, {}).update({f'Δauc {s}': r['Δauc'], f'Δll {s}': r['Δll(+better)'], f'se_ll {s}': r['se_ll']})
pd.DataFrame(rows).T.style.format('{:+.5f}')

out-of-fold R² of log price: 0.738; residual sd 0.444
main price drivers (gain share): {'CREDIT_SCORE': 0.65, 'STATE_CD': 0.122, 'CARRIER_STATE': 0.028, 'PKG_CAT': 0.008, 'VEH_SAFETY_MIN': 0.006}


,"Δauc (0,)","Δll (0,)","se_ll (0,)","Δauc (1, 2)","Δll (1, 2)","se_ll (1, 2)"
price → expected price only,-0.00305,-8.36454,+1.52248,-0.00313,-8.34780,+1.38199
price → residual only,-0.00354,-10.01855,+2.35117,-0.00371,-10.25435,+1.41210
+ log residual per era (competitiveness),-0.00024,-0.77215,+0.46815,-0.00022,-0.74319,+0.26320
+ expected price per era,-0.00003,+0.06024,+1.00886,+0.00007,+0.05318,+1.07916


**Reading:**
- **Price = credit-driven rating + state + substantial noise:** out-of-fold R² 0.74; credit carries ~65% of the explained part.
- **Conversion responds to the actual quoted price.** Using only the expected part or only the noise costs ~0.003 AUC, and a separate "competitiveness" term adds nothing. So there's no hidden price benchmark.

## 6. Prior carrier by state vs real NAIC market shares
- Source: NAIC Market Share Reports, Total Private Passenger Auto, top 10 groups per state, 2010 and 2017. These are user-downloaded PDFs in `data/external/manual/`, parsed by `lookup_tests.naic_top10`.
- Test: the best one-to-one assignment of Carrier_1–8 to real groups, minimizing the L1 distance between state share profiles.
- Null: the same with each carrier's state profile shuffled across states (1,000 times). If the labels were real insurers, the real assignment would fit far better than the null.

In [7]:
ours = (pd.crosstab(P.PRIOR_CARRIER_GRP, P.STATE_CD, normalize='columns') * 100)
display(ours.round(1))
ours8 = ours.loc[[f'Carrier_{k}' for k in range(1, 9)], ['AL', 'CT', 'FL', 'GA', 'MN', 'NJ', 'NY', 'WI']]
for yr in (2010, 2017):
    N = LT.naic_top10(yr); r = LT.carrier_match(N, ours8)
    print(f'{yr}: best L1 distance {r["cost"]:.0f} vs shuffled-null mean {r["null_mean"]:.0f} (5th pct {r["null_p05"]:.0f}); p = {r["p"]:.2f}')
    for c, g in r['assign'].items():
        print(f'   {c} ~ {g:32s} rank corr {pd.Series(ours8.loc[c].values).corr(pd.Series(r["R"].loc[g].values), method="spearman"):+.2f}')
    print('   real leader share by state:', N.groupby('ST').share.max().round(1).to_dict())
print('our largest carrier share by state:', ours8.max().round(1).to_dict())

STATE_CD,AL,CT,FL,GA,MN,NJ,NY,WI
PRIOR_CARRIER_GRP,,,,,,,,
Carrier_1,7.2,3.1,13.0,12.5,21.7,11.4,16.8,15.6
Carrier_2,4.6,18.4,18.8,13.9,2.1,2.0,4.2,13.6
Carrier_3,19.8,4.2,1.5,12.9,8.0,17.4,13.8,3.9
Carrier_4,7.8,13.1,4.2,11.7,14.7,10.6,3.7,2.7
Carrier_5,6.9,10.9,16.0,8.3,24.5,17.5,17.6,4.1
Carrier_6,14.3,11.0,5.0,9.0,2.6,23.3,12.9,17.7
Carrier_7,13.9,18.8,17.5,7.9,11.4,2.1,3.1,9.5
Carrier_8,6.3,0.8,4.9,11.2,3.6,5.0,15.3,15.9
Missing,9.1,10.5,9.8,10.3,9.2,10.6,10.1,10.7


2010: best L1 distance 368 vs shuffled-null mean 377 (5th pct 349); p = 0.29
   Carrier_1 ~ STATE FARM GRP                   rank corr +0.17
   Carrier_2 ~ TRAVELERS GRP                    rank corr +0.13
   Carrier_3 ~ ALLSTATE INS GRP                 rank corr +0.29
   Carrier_4 ~ ZURICH INS GRP                   rank corr +0.07
   Carrier_5 ~ BERKSHIRE HATHAWAY GRP           rank corr +0.31
   Carrier_6 ~ LIBERTY MUT GRP                  rank corr +0.02
   Carrier_7 ~ PROGRESSIVE GRP                  rank corr +0.31
   Carrier_8 ~ UNITED SERV AUTOMOBILE ASSN GRP  rank corr -0.07
   real leader share by state: {'AL': 23.9, 'CT': 13.1, 'FL': 20.8, 'GA': 22.1, 'MN': 22.0, 'NJ': 14.8, 'NY': 24.0, 'WI': 22.8}


2017: best L1 distance 389 vs shuffled-null mean 381 (5th pct 353); p = 0.69
   Carrier_1 ~ PROGRESSIVE GRP                  rank corr +0.69
   Carrier_2 ~ TRAVELERS GRP                    rank corr -0.08
   Carrier_3 ~ ALFA INS GRP                     rank corr +0.58
   Carrier_4 ~ LIBERTY MUT GRP                  rank corr +0.31
   Carrier_5 ~ BERKSHIRE HATHAWAY GRP           rank corr +0.31
   Carrier_6 ~ ALLSTATE INS GRP                 rank corr +0.38
   Carrier_7 ~ UNITED SERV AUTOMOBILE ASSN GRP  rank corr +0.33
   Carrier_8 ~ AMERICAN FAMILY INS GRP          rank corr +0.22
   real leader share by state: {'AL': 25.2, 'CT': 18.0, 'FL': 24.4, 'GA': 22.5, 'MN': 24.7, 'NJ': 20.6, 'NY': 31.9, 'WI': 18.7}
our largest carrier share by state: {'AL': 19.8, 'CT': 18.8, 'FL': 18.8, 'GA': 13.9, 'MN': 24.5, 'NJ': 23.3, 'NY': 17.6, 'WI': 17.7}


**Reading:** **no match.**
- The best assignment of our 8 carriers to real groups fits no better than shuffled profiles (p = 0.29 for 2010, 0.71 for 2017).
- In real markets State Farm or Berkshire (GEICO) lead almost every state (13–32%); our carriers have no consistent leader.
- Insurers outside the real top 10 hold 11–30% of each market, vs our 'Other' at 0.1–10%; 'Missing' is a flat ~10% everywhere.
- So the carrier-by-state table is synthetic (random shares per state).

## Conclusions
- **No real lookup found among the 12 variables.** Mileage, make/model, safety score, commute and marital status are independent draws. Education, living status and gender follow simple age/position rules.
- **Credit, price and CAT zone** come from synthetic state segment mixtures that contradict real state rankings.
- **Make/model names** come from a global vehicle-name list (with motorcycles and jet skis) but carry no information. The **luxury flag** we use is exactly the generator's.
- **Price** is rating factors plus noise (R² 0.74), and conversion uses the actual price, so there's no hidden competitiveness input.
- **Prior carrier** shares by state don't match real NAIC market shares in 2010 or 2017 (§6); they're random per-state tables.